In [0]:
%sql
-- CREACION DEL CATALOGO
CREATE CATALOG IF NOT EXISTS pf;

--CREACION DE LOS SCHEMAS NECESARIOS

CREATE SCHEMA IF NOT EXISTS pf.bronze;
CREATE SCHEMA IF NOT EXISTS pf.silver;
CREATE SCHEMA IF NOT EXISTS pf.gold;
CREATE SCHEMA IF NOT EXISTS pf.semantics;

-- VERIFICACION
SHOW SCHEMAS IN pf;


In [0]:
%sql

USE CATALOG pf;

CREATE SCHEMA IF NOT EXISTS pf.landing
COMMENT 'Esquema para almacenamiento';

CREATE VOLUME IF NOT EXISTS pf.landing.files
COMMENT 'Volume para almacenamiento de archivos .csv';

SHOW VOLUMES IN pf.landing;

In [0]:
%sql

-- =========================================================
-- DDL Control de Ingesta: hf.bronze.ingestion_control
-- Una fila por corrida. Sostiene el watermark incremental.
-- =========================================================

DROP TABLE IF EXISTS pf.bronze.ingestion_control;

CREATE TABLE IF NOT EXISTS pf.bronze.ingestion_control (
    run_id           STRING    NOT NULL COMMENT 'PK - ID de corrida',
    mode             STRING    COMMENT 'historical | incremental | full_refresh',
    start_ts         TIMESTAMP COMMENT 'Inicio de la ingesta',
    end_ts           TIMESTAMP COMMENT 'Fin de la ingesta',
    watermark_before TIMESTAMP COMMENT 'Watermark leido antes de correr',
    watermark_after  TIMESTAMP COMMENT 'Watermark calculado tras la corrida',
    n_pages          BIGINT    COMMENT 'Paginas descargadas',
    n_records        BIGINT    COMMENT 'Modelos descargados',
    status           STRING    COMMENT 'SUCCESS | PARTIAL | FAILED',
    notes            STRING    COMMENT 'Detalle / errores',
    PRIMARY KEY (run_id)
)
USING DELTA
TBLPROPERTIES (
   delta.enableChangeDataFeed = true,
   delta.autoOptimize.optimizeWrite = true,
   delta.autoOptimize.autoCompact = true,
   delta.feature.allowColumnDefaults = 'supported'
)
COMMENT 'Control de ingesta: watermark y resumen de cada corrida de la API';

SELECT run_id, mode, status, watermark_after, n_records
FROM pf.bronze.ingestion_control
ORDER BY start_ts DESC
LIMIT 10;